In [0]:
catalog = "meu_catalog"

silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Silver: {silver_schema}")
print(f"Gold: {gold_schema}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Conferência das tabelas Silver esperadas
display(
    spark.sql(f"SHOW TABLES IN {silver_schema}")
)

In [0]:

# REGRAS DE NEGÓCIO: dim_movies representa um registro por filme e preserva apenas metadados definidos no Star Schema do escopo.
tb_info = spark.table(f"{silver_schema}.tb_info_filmes")
tb_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
tb_metrics = spark.table(f"{silver_schema}.tb_metricas_engajamento")
tb_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
tb_genres = spark.table(f"{silver_schema}.tb_generos")
tb_people = spark.table(f"{silver_schema}.tb_pessoas_empresas")
tb_fx = spark.table(f"{silver_schema}.tb_cotacao_dolar")

for nome, df in {
    "tb_info_filmes": tb_info,
    "tb_financeiro_filmes": tb_fin,
    "tb_metricas_engajamento": tb_metrics,
    "tb_avaliacoes_usuarios": tb_reviews,
    "tb_generos": tb_genres,
    "tb_pessoas_empresas": tb_people,
    "tb_cotacao_dolar": tb_fx,
}.items():
    print(f"{nome}:")
    print(df.columns)

In [0]:
# REGRAS DE NEGÓCIO: dim_genres deve ser um catálogo único e deduplicado; a chave substituta permite separar a identidade analítica da chave natural da origem.
def dedup_por_filme(df):
    w = Window.partitionBy("id_filme").orderBy(F.col("id_filme"))
    return (
        df.withColumn("_rn_gold", F.row_number().over(w))
          .filter(F.col("_rn_gold") == 1)
          .drop("_rn_gold")
    )

info_1filme = dedup_por_filme(tb_info)
fin_1filme = dedup_por_filme(tb_fin)
metrics_1filme = dedup_por_filme(tb_metrics)

filmes_lancados = (
    info_1filme
    .filter(
        (F.col("status_filme") == "Lançado") &
        F.col("data_lancamento").isNotNull() &
        (F.col("data_lancamento") <= F.current_date())
    )
)

print(f"Filmes lançados considerados: {filmes_lancados.count()}")

In [0]:

# REGRAS DE NEGÓCIO: dim_people consolida atores, diretores e roteiristas; produtoras são tratadas separadamente em dim_companies conforme o modelo dimensional.
dim_movies = (
    info_1filme
    .select(
        F.xxhash64(F.col("id_filme")).alias("sk_movie_id"),
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse"),
    )
    .dropDuplicates(["id_filme"])
)

(
    dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_movies")
)

display(spark.table(f"{gold_schema}.dim_movies").limit(10))

In [0]:
# REGRAS DE NEGÓCIO: dim_companies mantém um catálogo único de produtoras/estúdios, evitando repetição de nomes na camada analítica.
genres_clean = (
    tb_genres
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.trim(F.col("nome_genero").cast("string")).alias("nome_genero")
    )
    .filter(
        F.col("id_filme").isNotNull() &
        F.col("nome_genero").isNotNull() &
        (F.col("nome_genero") != "")
    )
    .dropDuplicates(["id_filme", "nome_genero"])
)

dim_genres = (
    genres_clean
    .select("nome_genero")
    .dropDuplicates()
    .withColumn("sk_genre_id", F.xxhash64(F.col("nome_genero")))
    .select("sk_genre_id", "nome_genero")
)

bridge_movie_genre = (
    genres_clean
    .join(dim_genres, on="nome_genero", how="inner")
    .join(
        dim_movies.select("sk_movie_id", "id_filme"),
        on="id_filme",
        how="inner"
    )
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)

In [0]:
# REGRAS DE NEGÓCIO: Tabelas-ponte são obrigatórias para relações muitos-para-muitos e impedem que múltiplos gêneros/pessoas/produtoras multipliquem o grão da fato.
from pyspark.sql import functions as F

dim_movies_gold = spark.table(f"{gold_schema}.dim_movies")
dim_genres_gold = spark.table(f"{gold_schema}.dim_genres")
tb_genres_gold = spark.table(f"{silver_schema}.tb_generos")

genres_clean = (
    tb_genres_gold
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.trim(F.col("nome_genero").cast("string")).alias("nome_genero")
    )
    .filter(
        F.col("id_filme").isNotNull() &
        F.col("nome_genero").isNotNull() &
        (F.col("nome_genero") != "")
    )
    .dropDuplicates(["id_filme", "nome_genero"])
)

bridge_movie_genre = (
    genres_clean.alias("g")
    .join(
        dim_movies_gold.select(
            "sk_movie_id",
            "id_filme"
        ).alias("m"),
        F.col("g.id_filme") == F.col("m.id_filme"),
        "inner"
    )
    .join(
        dim_genres_gold.select(
            "sk_genre_id",
            "nome_genero"
        ).alias("d"),
        F.col("g.nome_genero") == F.col("d.nome_genero"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id"),
        F.col("d.sk_genre_id")
    )
    .dropDuplicates()
)

print(f"Registros da bridge: {bridge_movie_genre.count():,}")

(
    bridge_movie_genre
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")
)

print("Tabela gold.bridge_movie_genre criada com sucesso.")

display(
    spark.table(f"{gold_schema}.bridge_movie_genre").limit(20)
)

In [0]:
# REGRAS DE NEGÓCIO: Avaliações são agregadas por filme porque dim_reviews deve representar uma métrica resumida, e não cada comentário individual.
people_clean = (
    tb_people
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.trim(F.col("nome_entidade").cast("string")).alias("nome_entidade"),
        F.trim(F.col("tipo_entidade").cast("string")).alias("tipo_entidade")
    )
    .filter(
        F.col("id_filme").isNotNull() &
        F.col("nome_entidade").isNotNull() &
        (F.col("nome_entidade") != "") &
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista", "Produtora")
    )
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

people_only = (
    people_clean.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("nome_entidade", "tipo_entidade").dropDuplicates()
)

dim_people = (
    people_only
    .withColumn("sk_person_id", F.xxhash64(F.col("nome_entidade"), F.col("tipo_entidade")))
    .select("sk_person_id", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
)

companies_only = people_clean.filter(F.col("tipo_entidade") == "Produtora").select("nome_entidade").dropDuplicates()

dim_companies = (
    companies_only.withColumn("sk_company_id", F.xxhash64(F.col("nome_entidade")))
    .select("sk_company_id", F.col("nome_entidade").alias("nome_produtora"))
)

for df, nome in [(dim_people, "dim_people"), (dim_companies, "dim_companies")]:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{gold_schema}.{nome}"))

display(spark.table(f"{gold_schema}.dim_people").limit(20))
display(spark.table(f"{gold_schema}.dim_companies").limit(20))

In [0]:
# REGRAS DE NEGÓCIO: fact_movies_performance deve ter grão de um registro por filme lançado. Os joins usam dimensões/agrupamentos de um filme para evitar duplicação de métricas.
bridge_movie_person = (
    people_clean.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        dim_people.select(
            "sk_person_id",
            F.col("nome_pessoa").alias("nome_entidade"),
            F.col("tipo_pessoa").alias("tipo_entidade")
        ),
        on=["nome_entidade", "tipo_entidade"], how="inner"
    )
    .select("sk_movie_id", "sk_person_id").dropDuplicates()
)

bridge_movie_company = (
    people_clean.filter(F.col("tipo_entidade") == "Produtora")
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        dim_companies.select("sk_company_id", F.col("nome_produtora").alias("nome_entidade")),
        on="nome_entidade", how="inner"
    )
    .select("sk_movie_id", "sk_company_id").dropDuplicates()
)

for df, nome in [(bridge_movie_person, "bridge_movie_person"), (bridge_movie_company, "bridge_movie_company")]:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{gold_schema}.{nome}"))

display(spark.table(f"{gold_schema}.bridge_movie_person").limit(20))
display(spark.table(f"{gold_schema}.bridge_movie_company").limit(20))

In [0]:
# REGRAS DE NEGÓCIO: O contexto para RAG deve ser uma frase corrida. Atores são agregados e campos potencialmente nulos recebem fallbacks para evitar que concatenação retorne NULL para o documento inteiro.
reviews_by_movie = (
    tb_reviews.groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)

dim_reviews = (
    reviews_by_movie
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .select(
        F.xxhash64(F.col("sk_movie_id")).alias("sk_review_id"),
        F.col("sk_movie_id"),
        F.col("qtd_avaliacoes_usuarios"),
        F.col("nota_media_usuarios")
    )
)

(dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{gold_schema}.dim_reviews"))

display(spark.table(f"{gold_schema}.dim_reviews").limit(20))

In [0]:
# REGRAS DE NEGÓCIO: A auditoria confirma o grão da fato e a existência das tabelas Gold exigidas antes das consultas de negócio.
fact_movies_performance = (
    filmes_lancados
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        fin_1filme.select("id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl"),
        on="id_filme", how="left"
    )
    .join(
        metrics_1filme.select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"),
        on="id_filme", how="left"
    )
    .select(
        F.col("sk_movie_id"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
    .dropDuplicates(["sk_movie_id"])
)

(fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{gold_schema}.fact_movies_performance"))

display(spark.table(f"{gold_schema}.fact_movies_performance").limit(10))

In [0]:
atores = (
    bridge_movie_person.alias("b")
    .join(dim_people.alias("p"), F.col("b.sk_person_id") == F.col("p.sk_person_id"), "inner")
    .filter(F.col("p.tipo_pessoa") == "Ator")
    .groupBy("b.sk_movie_id")
    .agg(F.concat_ws(", ", F.array_sort(F.collect_set(F.col("p.nome_pessoa")))).alias("atores_principais"))
)

diretores = (
    bridge_movie_person.alias("b")
    .join(dim_people.alias("p"), F.col("b.sk_person_id") == F.col("p.sk_person_id"), "inner")
    .filter(F.col("p.tipo_pessoa") == "Diretor")
    .groupBy("b.sk_movie_id")
    .agg(F.concat_ws(", ", F.array_sort(F.collect_set(F.col("p.nome_pessoa")))).alias("diretor"))
)

genai_context = (
    dim_movies.alias("m")
    .join(fact_movies_performance.alias("f"), on="sk_movie_id", how="left")
    .join(atores.alias("a"), on="sk_movie_id", how="left")
    .join(diretores.alias("d"), on="sk_movie_id", how="left")
    .select(
        F.col("m.id_filme").alias("movie_id"),
        F.col("m.titulo").alias("title"),
        F.concat(
            F.lit("O filme "),
            F.coalesce(F.col("m.titulo"), F.lit("Título não informado")),
            F.lit(", lançado no ano de "),
            F.coalesce(F.col("m.ano_lancamento").cast("string"), F.lit("ano não informado")),
            F.lit(", faturou "),
            F.coalesce(F.col("f.receita_usd").cast("string"), F.lit("receita não informada")),
            F.lit(" e teve um custo de "),
            F.coalesce(F.col("f.orcamento_usd").cast("string"), F.lit("orçamento não informado")),
            F.lit(". Estrelado por "),
            F.coalesce(F.col("a.atores_principais"), F.lit("elenco não informado")),
            F.lit(" e dirigido por "),
            F.coalesce(F.col("d.diretor"), F.lit("diretor não informado")),
            F.lit(", o filme possui a seguinte sinopse: "),
            F.coalesce(F.when(F.trim(F.col("m.sinopse")) != "", F.col("m.sinopse")), F.lit("Sinopse não informada.")),
            F.lit(".")
        ).alias("llm_context_document")
    )
)

(genai_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{gold_schema}.gold_genai_movies_context"))

display(spark.table(f"{gold_schema}.gold_genai_movies_context").limit(10))


In [0]:
tabelas_gold = [
    "fact_movies_performance", "dim_movies", "dim_genres", "dim_people",
    "dim_companies", "dim_reviews", "bridge_movie_genre",
    "bridge_movie_person", "bridge_movie_company", "gold_genai_movies_context"
]

for tabela in tabelas_gold:
    qtd = spark.table(f"{gold_schema}.{tabela}").count()
    print(f"{tabela}: {qtd:,} registros")

qtd_fato = spark.table(f"{gold_schema}.fact_movies_performance").count()
qtd_fato_distinto = spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id").distinct().count()

print(f"\nGrão da fato: {qtd_fato:,} linhas")
print(f"Filmes distintos na fato: {qtd_fato_distinto:,}")

if qtd_fato != qtd_fato_distinto:
    raise ValueError("ERRO: a fact_movies_performance possui mais de uma linha por filme.")

print("OK: a fato mantém um único registro por filme.")

Desafio

In [0]:
# P1. RECEITA TOTAL EM BRL
p1 = spark.sql(f"""
    SELECT ROUND(SUM(receita_brl), 2) AS receita_total_brl
    FROM {gold_schema}.fact_movies_performance
""")
display(p1)


In [0]:
# P2. TOP 5 POPULARIDADE
p2 = spark.sql(f"""
    SELECT m.titulo, f.popularidade
    FROM {gold_schema}.fact_movies_performance f
    INNER JOIN {gold_schema}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
""")
display(p2)

In [0]:
# P3. FILMES POR GÊNERO
p3 = spark.sql(f"""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM {gold_schema}.bridge_movie_genre b
    INNER JOIN {gold_schema}.dim_genres g ON b.sk_genre_id = g.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
""")
display(p3)


In [0]:
# P4. TOP 10 RECEITA + RANK
p4 = spark.sql(f"""
    WITH ranking AS (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM {gold_schema}.fact_movies_performance f
        INNER JOIN {gold_schema}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    SELECT titulo, receita_usd, receita_brl, posicao_ranking
    FROM ranking
    ORDER BY posicao_ranking, titulo
    LIMIT 10
""")
display(p4)

In [0]:
# P5. ATOR COM MAIS PARTICIPAÇÕES NOS ÚLTIMOS 2 ANOS
p5 = spark.sql(f"""
    WITH data_referencia AS (
        SELECT MAX(data_lancamento) AS data_maxima
        FROM {gold_schema}.dim_movies
        WHERE status_filme = 'Lançado'
          AND data_lancamento IS NOT NULL
          AND data_lancamento <= CURRENT_DATE()
    ),
    filmes_periodo AS (
        SELECT sk_movie_id
        FROM {gold_schema}.dim_movies
        CROSS JOIN data_referencia
        WHERE status_filme = 'Lançado'
          AND data_lancamento BETWEEN ADD_MONTHS(data_maxima, -24) AND data_maxima
    )
    SELECT p.nome_pessoa, COUNT(DISTINCT b.sk_movie_id) AS qtd_participacoes
    FROM filmes_periodo f
    INNER JOIN {gold_schema}.bridge_movie_person b ON f.sk_movie_id = b.sk_movie_id
    INNER JOIN {gold_schema}.dim_people p ON b.sk_person_id = p.sk_person_id
    WHERE p.tipo_pessoa = 'Ator'
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC, p.nome_pessoa
    LIMIT 1
""")
display(p5)

In [0]:
# P6. PRODUTORA COM MAIOR LUCRO NOS ÚLTIMOS 5 ANOS
p6 = spark.sql(f"""
    WITH data_referencia AS (
        SELECT MAX(data_lancamento) AS data_maxima
        FROM {gold_schema}.dim_movies
        WHERE status_filme = 'Lançado'
          AND data_lancamento IS NOT NULL
          AND data_lancamento <= CURRENT_DATE()
    ),
    filmes_periodo AS (
        SELECT sk_movie_id
        FROM {gold_schema}.dim_movies
        CROSS JOIN data_referencia
        WHERE status_filme = 'Lançado'
          AND data_lancamento BETWEEN ADD_MONTHS(data_maxima, -60) AND data_maxima
    )
    SELECT c.nome_produtora,
           ROUND(SUM(f.lucro_usd), 2) AS lucro_total_usd,
           ROUND(SUM(f.lucro_brl), 2) AS lucro_total_brl
    FROM filmes_periodo m
    INNER JOIN {gold_schema}.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    INNER JOIN {gold_schema}.bridge_movie_company b ON m.sk_movie_id = b.sk_movie_id
    INNER JOIN {gold_schema}.dim_companies c ON b.sk_company_id = c.sk_company_id
    WHERE f.lucro_usd IS NOT NULL OR f.lucro_brl IS NOT NULL
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC NULLS LAST, c.nome_produtora
    LIMIT 1
""")
display(p6)

In [0]:
context_audit = spark.sql(f"""
    SELECT
        COUNT(*) AS total_registros,
        SUM(CASE WHEN movie_id IS NULL THEN 1 ELSE 0 END) AS movie_id_nulos,
        SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS title_nulos,
        SUM(CASE WHEN llm_context_document IS NULL THEN 1 ELSE 0 END) AS contexto_nulos
    FROM {gold_schema}.gold_genai_movies_context
""")

display(context_audit)
display(spark.sql(f"SHOW TABLES IN {gold_schema}"))